# 03. Dynare Compatibility in `dyno.py`

[Dynare](https://www.dynare.org/) is the widely used standard in central banks and academia for macroeconomic DSGE modeling. However, Dynare typically requires Matlab or Octave and external preprocessors.

`dyno.py` provides native parsing and execution of Dynare `.mod` files directly in Python!

In this notebook, we cover:
1. **Loading a `.mod` File**: Seamless import with `DynoModel("models/rbc.mod")`.
2. **Syntax Comparison**: How `.mod` syntax maps to `.dyno` syntax.
3. **Solving & Simulating**: Computing decision rules and IRFs from a `.mod` model.
4. **Leveraging Python**: Combining Dynare models with the modern Python data science ecosystem.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dyno import DynoModel, simulate, irfs

pd.set_option('display.precision', 4)


## 1. Loading a Dynare `.mod` File

Let's load `models/rbc.mod` directly using `DynoModel`:


In [2]:
# Load standard Dynare .mod file
mod_model = DynoModel("models/rbc.mod")

print(f"Model name      : {mod_model.name}")
print(f"Endogenous vars : {mod_model.symbols['endogenous']}")
print(f"Exogenous vars  : {mod_model.symbols['exogenous']}")
print(f"Parameters      : {mod_model.symbols['parameters']}")


Model name      : rbc
Endogenous vars : ['r', 'n', 'k', 'y', 'i', 'c', 'w', 'a']
Exogenous vars  : ['epsilon']
Parameters      : ['beta', 'delta', 'nss', 'eta', 'alpha', 'rho', 'chi']


## 2. Syntax Comparison: Dyno vs. Dynare

| Feature | Dynare (`.mod`) | Dyno (`.dyno`) |
| :--- | :--- | :--- |
| **Variable Declaration** | `var c, k, y;` | Inferred automatically from equations |
| **Shocks Declaration** | `varexo epsilon;` | Inferred from shock assignments |
| **Parameter Assignment**| `beta = 0.985;` | `beta <- 0.985` |
| **Time Indexing** | `k(-1)`, `c(+1)`, `c` | `k[t-1]`, `c[t+1]`, `c[t]` |
| **Equations Block** | `model; 1/c = ...; end;` | Direct equation list (`1/c[t] = ...`) |
| **Steady State** | `steady_state_model; ... end;` | `k[~] <- ...` or `c[~] <- ...` |
| **Shock Distribution** | `shocks; var epsilon; stderr 0.009; end;` | `epsilon[t] <- N(0, 0.009^2)` |

Dyno parses Dynare's `var`, `varexo`, `parameters`, `model;`, `steady_state_model;`, and `shocks;` blocks seamlessly.


## 3. Steady State and Blanchard-Kahn Verification

Let's check the steady-state solution and eigenvalue conditions computed from the `.mod` file:


In [3]:
# Display steady state
ss_df = pd.DataFrame.from_dict(mod_model.steady_state, orient='index', columns=['Steady State'])
display(ss_df)

# Run Blanchard-Kahn check
mod_model.check()


,Steady State
a,1.0000
r,0.0402
n,0.3300
k,6.9655
y,0.8757
w,1.8044
i,0.1741
c,0.7015
epsilon,0.0000


equations,8,
variables,9,
endogenous,8,"r, n, k, y, i, c, w, a"
exogenous,1,epsilon
constants,7,"beta, delta, nss, eta, alpha, rho, chi"


## 4. Solving the Dynare Model

Now we solve for the recursive decision rule and policy functions:


In [4]:
solution = mod_model.solve()
dr = solution.decision_rule

# Display policy function coefficients (Jacobian)
dr.coefficients_as_df()


(        r     n       k       y       i       c       w    a
 0  0.0402  0.33  6.9655  0.8757  0.1741  0.7015  1.8044  1.0,
       r[t-1]  n[t-1]  k[t-1]  y[t-1]  i[t-1]  c[t-1]  w[t-1]  a[t-1]  \
 r[t]     0.0     0.0 -0.0049     0.0     0.0     0.0     0.0  0.0560   
 n[t]     0.0     0.0 -0.0122     0.0     0.0     0.0     0.0  0.2387   
 k[t]     0.0     0.0  0.9398     0.0     0.0     0.0     0.0  0.9997   
 y[t]     0.0     0.0  0.0182     0.0     0.0     0.0     0.0  1.2188   
 i[t]     0.0     0.0 -0.0352     0.0     0.0     0.0     0.0  0.9997   
 c[t]     0.0     0.0  0.0533     0.0     0.0     0.0     0.0  0.2191   
 w[t]     0.0     0.0  0.1043     0.0     0.0     0.0     0.0  1.2063   
 a[t]     0.0     0.0  0.0000     0.0     0.0     0.0     0.0  0.9000   
 
       epsilon[t]  
 r[t]      0.0622  
 n[t]      0.2652  
 k[t]      1.1107  
 y[t]      1.3542  
 i[t]      1.1107  
 c[t]      0.2434  
 w[t]      1.3404  
 a[t]      1.0000  )

## 5. Interactive Impulse Response Functions

Let's compute the IRFs and render them using Plotly:


In [5]:
# Compute IRFs (log-deviation)
irf_dict = solution.irfs(type="log-deviation", T=40)
tfp_irf = irf_dict["epsilon"]
display(tfp_irf.head())

# Plot interactive IRFs
fig = solution.plot(type="log-deviation")
fig.update_layout(height=500, width=800, title="Dynare Model: IRFs to TFP Shock (log-deviation)")
fig.show()


,r,n,k,y,i,c,w,a
0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,1.3918,0.7233,0.1435,1.3918,5.7407,0.3123,0.6686,0.9000
2,1.1299,0.6139,0.2641,1.2734,4.9649,0.3571,0.6595,0.8100
3,0.9015,0.5178,0.3644,1.1656,4.2787,0.3928,0.6478,0.7290
4,0.7029,0.4333,0.4471,1.0673,3.6726,0.4207,0.6341,0.6561


## 6. Seamless Integration with Python Ecosystem

Unlike traditional Dynare workflows that produce `.mat` files or raw command-line text, Dyno returns native **Pandas DataFrames** and **NumPy arrays**.

You can immediately perform custom downstream analysis:


In [6]:
# 1. Stochastic simulation
sim_data = simulate(solution, T=200)

# 2. Compute summary statistics (volatilities relative to output)
std_y = sim_data['y'].std()
relative_vol = sim_data.std() / std_y

# 3. Compute cross-correlations with output
corrs = sim_data.corrwith(sim_data['y'])

summary_table = pd.DataFrame({
    'Std. Dev.': sim_data.std(),
    'Relative Vol (σ_x / σ_y)': relative_vol,
    'Corr with Output': corrs
})
summary_table


,Std. Dev.,Relative Vol (σ_x / σ_y),Corr with Output
r,0.0011,0.0420,0.6916
n,0.0045,0.1711,0.8748
k,0.1586,6.0025,0.5874
y,0.0264,1.0000,1.0000
i,0.0191,0.7233,0.9316
c,0.0111,0.4190,0.7785
w,0.0349,1.3222,0.9394
a,0.0186,0.7027,0.9950
